# 020 — Vectorless RAG
**Advanced RAG Series** | RAG without any embeddings

Covers: BM25-only pipeline · TF-IDF retrieval · Full RAG with zero vector math


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(f"Missing: {path}\nRun python create_datasets.py from project root.")

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars across 4 files")


✓ Loaded 22,486 chars across 4 files


In [4]:
# ── 1. Chunk the corpus into passages ───────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
chunks = splitter.split_text(all_text)
print(f"Total chunks: {len(chunks)}")
print("\nSample chunk:")
print(chunks[10][:300])


Total chunks: 85

Sample chunk:
Unsupervised learning is a type of algorithm that learns patterns from untagged data. The hope
is that through mimicry, which is computationally cheaper than other approaches, the machine
is forced to build a compact internal representation of its world.


In [5]:
# ── 2. Build BM25 index (no embeddings, just word counts) ───────────────
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

tokenized_chunks = [tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"✓ BM25 index built over {len(chunks)} chunks")
print(f"  Average tokens per chunk: {sum(len(t) for t in tokenized_chunks) // len(tokenized_chunks)}")


✓ BM25 index built over 85 chunks
  Average tokens per chunk: 39


In [6]:
# ── 3. BM25 retrieval function ──────────────────────────────────────────
def bm25_retrieve(query: str, top_k: int = 5):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[-top_k:][::-1]
    return [(chunks[i], float(scores[i])) for i in top_idx]

# Test retrieval
query = "what is retrieval augmented generation"
results = bm25_retrieve(query, top_k=3)
print(f"Query: {query!r}\n")
for i, (chunk, score) in enumerate(results, 1):
    print(f"[{i}] score={score:.3f}")
    print(chunk[:200])
    print()


Query: 'what is retrieval augmented generation'

[1] score=12.006
Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines information
retrieval with text generation. Rather than relying solely on knowledge

[2] score=7.847
RAG Evaluation

RAGAS (Retrieval-Augmented Generation Assessment) provides automated metrics:
- Faithfulness: is the answer supported by the retrieved context?
- Answer relevancy: does the answer addr

[3] score=5.370
Recurrent Neural Networks for NLP

RNNs process sequences by maintaining a hidden state that captures information about previous
tokens. Long Short-Term Memory (LSTM) networks address the vanishing gr



In [7]:
# ── 4. Full vectorless RAG pipeline ─────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer ONLY using the context below.\n"
    "If the answer is not in the context, say \"I don't know\".\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

def vectorless_rag(question: str, top_k: int = 5) -> str:
    retrieved = bm25_retrieve(question, top_k)
    context = "\n\n---\n\n".join([f"[Score: {s:.2f}]\n{c}" for c, s in retrieved])
    chain = RAG_PROMPT | llm
    response = chain.invoke({"context": context, "question": question})
    return response.content

answer = vectorless_rag("How does RAG reduce hallucination in language models?")
print(answer)


RAG reduces hallucination in language models by giving them access to up-to-date information and verifiable sources, addressing the key limitation of knowledge staleness and hallucination.


In [8]:
# ── 5. Vectorless vs Vector RAG — when to use which ─────────────────────
print("VECTORLESS RAG (BM25 only)")
print("  + No GPU, no embedding model, no vector DB needed")
print("  + Instant startup, tiny memory footprint")
print("  + Perfect for exact keywords: product codes, names, dates")
print("  - Misses synonyms: 'car' won't match 'automobile'")
print("  - Misses paraphrased queries")
print()
print("VECTOR RAG (dense embeddings)")
print("  + Understands semantic meaning")
print("  + Handles paraphrasing and synonyms")
print("  - Requires embedding model + vector DB")
print("  - Can miss exact rare terms (product IDs, model names)")
print()
print("BEST PRACTICE: Hybrid (BM25 + Dense) — see notebook 006")

# Quick benchmark: time a BM25 query
import time
t0 = time.time()
for _ in range(100):
    bm25_retrieve("transformer attention mechanism", top_k=5)
elapsed = (time.time() - t0) * 10  # ms per query
print(f"\nBM25 average latency: {elapsed:.2f} ms/query (100 runs)")


VECTORLESS RAG (BM25 only)
  + No GPU, no embedding model, no vector DB needed
  + Instant startup, tiny memory footprint
  + Perfect for exact keywords: product codes, names, dates
  - Misses synonyms: 'car' won't match 'automobile'
  - Misses paraphrased queries

VECTOR RAG (dense embeddings)
  + Understands semantic meaning
  + Handles paraphrasing and synonyms
  - Requires embedding model + vector DB
  - Can miss exact rare terms (product IDs, model names)

BEST PRACTICE: Hybrid (BM25 + Dense) — see notebook 006

BM25 average latency: 0.05 ms/query (100 runs)


In [9]:
# ── 6. TF-IDF retrieval (sklearn alternative, no extra deps) ─────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(chunks)
print(f"TF-IDF matrix: {tfidf_matrix.shape[0]} docs × {tfidf_matrix.shape[1]} terms")

def tfidf_retrieve(query: str, top_k: int = 5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix)[0]
    top_idx = scores.argsort()[-top_k:][::-1]
    return [(chunks[i], float(scores[i])) for i in top_idx]

# Compare BM25 vs TF-IDF on same query
q = "neural network training backpropagation"
bm25_results  = bm25_retrieve(q, top_k=3)
tfidf_results = tfidf_retrieve(q, top_k=3)

print(f"\nQuery: {q!r}")
print("\nBM25 top-1:")
print(bm25_results[0][0][:200])
print("\nTF-IDF top-1:")
print(tfidf_results[0][0][:200])


TF-IDF matrix: 85 docs × 1023 terms

Query: 'neural network training backpropagation'

BM25 top-1:
Backpropagation is the algorithm used to compute gradients in neural networks. It applies the
chain rule of calculus to propagate error signals backwards through the network, computing
how much each w

TF-IDF top-1:
Backpropagation is the algorithm used to compute gradients in neural networks. It applies the
chain rule of calculus to propagate error signals backwards through the network, computing
how much each w
